# Track 1 · P04: classifier metrics by object size, and the test label fix (report only)

AI251P Capstone, Phase 2, P04. Owner: Abdulmohsen Alharthi. Due 10 Nov 2026 (target 3 Nov).

**The model does not change.** `cls-full-64-s43-lr0.002` (the checkpoint delivered to integration as ONNX `2026-10-31_ClsResNet18_s43_v1`), **raw logits, no logit bias.** Part (c) (more truck/bus examples, retraining) is cancelled and goes into the report as future work.

- **Part 1, validation only.** Reproduce val macro-F1 (a check), then metrics by object size: `size = sqrt(w * h)` in original-frame pixels from CropIndex; COCO bins small < 32, medium 32 to < 96, large >= 96. Per size bin and dataset: macro-F1, per-class recall / F1 / support, truck→car and bus→car rates. A class with fewer than 30 crops in a cell is shown as `n<30` (never dropped). `size_bins_val.csv` is written **before** test is opened.
- **Part 2, one opening of test, for the report.** A preflight prints exactly what will be computed and checks that the code, the weights and the label-fix list are fixed; then one `TestLog.csv` entry; then the same tables on test, twice: **original labels (the headline)** and corrected labels. **Nothing is tuned, thresholded or decided on test.** If anything fails after the TestLog entry, the opening still counts and a re-run refuses (the reason is already logged).
- **Label fix:** the test crops that the Step 4 label audit (blind, done before any model prediction) judged `wrong_class` (relabelled to `true_label`) or `not_object` (removed from the corrected numbers). A crop found from the Step 8 error gallery (after seeing predictions) can only enter through `GALLERY_FIX` and is marked as biased upward in the report. No list found -> the notebook stops.
- Guard: every file opened under the Drive folder whose path contains `Test` or `AlHada` stops the notebook, except `TestLog.csv` (always) and the test crops / `TestPredictions_v1` (only after the TestLog entry). Al-Hada is never read.
- One Drive root `DRIVE_PROJECT`; every path in an output is relative to it. Output `02_Classification/P04/<YYYY-MM-DD>_SizeBins_vN/`, written as `..._INCOMPLETE` and renamed only when every check passes (never overwritten).
- Copied verbatim: `class Pretrained`, `MEAN`/`STD`, `gpu_eval` (from `notebooks/track1/train_cls_2026-10-25.ipynb`, as in `export_onnx_2026-10-30.ipynb`). Inference in fp32, TF32 off.

Runtime: Colab, T4 GPU (CPU works, slower). **Run all once**; clear outputs before uploading to the repo.

## 0. Settings, mount Drive

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'   # the only Drive path

IN = {   # relative to DRIVE_PROJECT, read-only; nothing here may contain 'Test' or 'AlHada'
    'ckpt':       '02_Classification/Checkpoints/2026-10-23_cls-full-64-s43-lr0.002_v4.pt',
    'val_npy':    '01_Data/Processed/Crops/2026-10-20_CropsVal_v1.npy',
    'crop_index': '01_Data/Processed/Crops/2026-10-20_CropIndex_v1.csv.gz',
    'audit_dir':  '01_Data/Processed/Audit',                   # Step 4: *_AuditFilled_vN.csv, the latest is used (as Step 4 does)
}
IN_TEST = {   # opened only in Part 2: TestLog from the start, the other two only after the TestLog entry
    'test_log':  '02_Classification/TestLog.csv',
    'test_npy':  '01_Data/Processed/Crops/2026-10-20_CropsTest_v1.npy',
    'test_pred': '02_Classification/2026-10-23_TestPredictions_v1.csv.gz',   # Step 8 predictions, column fullFT_s43_raw
}
GALLERY_FIX = None   # only if decided: CSV under DRIVE_PROJECT with crop_id, corrected_label, how (found AFTER seeing predictions)
CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']
EXPECT = {'img_size': 64, 'val_crops': 13_800, 'crop_side': 96,
          'val_macro_f1': 0.8233, 'val_macro_f1_tol': 0.0005,     # fp32 val macro-F1 of this checkpoint (export notebook)
          'test_log_entries_before': 2}                           # 23 Oct step8, 25 Oct step8b
SIZE_EDGES = (32, 96)            # COCO: small < 32 <= medium < 96 <= large, size = sqrt(w * h) px in the original frame
BINS = ['small', 'medium', 'large']
MIN_N = 30                       # a class with fewer crops than this in a cell is shown as 'n<30'
FOCUS = [('truck', 'car'), ('bus', 'car')]
EXCLUDE_RGB = ('SeaDronesSee', 'rgb')   # 25 Oct decision: the primary test number is without SDS source 'rgb'
TIE_GAP = 1e-2                   # as step8b: argmax may differ from TestPredictions_v1 (fp16) only if top1 - top2 < TIE_GAP
BATCH = 512
NOTEBOOK = 'p04_size_bins_label_fix_2026-11-01'
REASON = 'P04 report-only: size bins (COCO 32/96) and audit label fix, cls-full-64-s43-lr0.002 raw logits, no tuning'
FORBIDDEN = ('Test', 'AlHada')
assert not [v for v in IN.values() if any(f in v for f in FORBIDDEN)], 'an input path points at test or Al-Hada'
assert not any('AlHada' in v for v in IN_TEST.values())

from google.colab import drive
drive.mount('/content/drive')

## 1. Guard, inputs, check log

From here on every `open()` of a file under the Drive folder (this includes `np.load`, `pd.read_csv`, `torch.load`, `savefig`) goes through a guard. A path containing `Test` or `AlHada` stops the notebook unless it is in `ALLOWED`, which holds only `TestLog.csv` until the TestLog entry is written.

In [ ]:
import os, io, re, csv, json, time, hashlib, builtins
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
import matplotlib, matplotlib.pyplot as plt
from sklearn.metrics import f1_score, recall_score
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 400)

ROOT = Path(DRIVE_PROJECT)
assert ROOT.is_dir(), f'DRIVE_PROJECT not found: {DRIVE_PROJECT}'
ALLOWED = {os.path.abspath(ROOT / IN_TEST['test_log'])}
_builtin_open = getattr(builtins.open, '_unguarded', builtins.open)
def _guarded_open(file, *args, **kwargs):
    if isinstance(file, (str, bytes, os.PathLike)):
        s = os.fsdecode(file)
        if s.startswith(('/content/drive', DRIVE_PROJECT)) and any(f in s for f in FORBIDDEN) \
           and (os.path.abspath(s) not in ALLOWED or 'AlHada' in s):
            raise PermissionError(f'forbidden path (test / Al-Hada) opened: {s}')
    return _builtin_open(file, *args, **kwargs)
_guarded_open._unguarded = _builtin_open
builtins.open = io.open = _guarded_open

P = {k: ROOT / v for k, v in {**IN, **IN_TEST}.items()}
missing = [f'{k}: {v}' for k, v in {**IN, **IN_TEST}.items() if not (P[k].is_dir() if k == 'audit_dir' else P[k].is_file())]
if GALLERY_FIX is not None and not (ROOT / GALLERY_FIX).is_file(): missing.append(f'GALLERY_FIX: {GALLERY_FIX}')
if missing:
    print('MISSING INPUTS (nothing was read or written):'); print('\n'.join('  ' + m for m in missing))
    raise FileNotFoundError(f'{len(missing)} missing input(s), see the list above')
_aud0 = [f for f in os.listdir(P['audit_dir']) if re.fullmatch(r'\d{4}-\d{2}-\d{2}_AuditFilled_v\d+\.csv', f)]
if not _aud0: raise FileNotFoundError(f'no *_AuditFilled_vN.csv in {IN["audit_dir"]}: no label-fix list (nothing was written)')
# the full list is read and printed in the preflight (section 8); here only: does the latest sheet have test rows at all?
_aud0 = sorted(_aud0, key=lambda f: (f[:10], int(re.search(r'_v(\d+)\.csv$', f)[1])))[-1]
if not (pd.read_csv(P['audit_dir'] / _aud0, keep_default_na=False, dtype=str).split == 'test').any():
    raise RuntimeError(f'{_aud0} has no test rows: no label-fix list, stop and tell Abdulmohsen (nothing was written)')
LOG_COLS = ['when', 'checkpoints', 'reason', 'notebook_version']   # the columns this notebook fills; none is ever added
_log0 = pd.read_csv(P['test_log'], dtype=str, keep_default_na=False)   # allowed path
if [c for c in LOG_COLS if c not in _log0.columns]:                   # stop before anything is written (and before test)
    print('TestLog.csv columns:', list(_log0.columns), '| missing:', [c for c in LOG_COLS if c not in _log0.columns])
    raise RuntimeError('TestLog.csv header lacks a column this notebook fills; no column is added: stop (nothing was written)')
assert REASON not in set(_log0['reason']), 'this P04 opening of test is already in TestLog.csv: the notebook is not run twice'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None
torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False   # plain fp32 everywhere
torch.backends.cudnn.benchmark = False
import sklearn, PIL
VERSIONS = {'python': os.sys.version.split()[0], 'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'matplotlib': matplotlib.__version__,
            'cuda (torch)': torch.version.cuda, 'gpu': GPU}
print(json.dumps(VERSIONS, indent=1))

REPORT = []
def check(name, expected, actual, ok=None):
    """hard assert; every check is logged as expected / actual / pass"""
    ok = (expected == actual) if ok is None else bool(ok)
    j = lambda v: v if isinstance(v, (str, int, float, bool, type(None), list, dict)) else str(v)
    REPORT.append({'check': name, 'expected': j(expected), 'actual': j(actual), 'pass': ok})
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected {expected}, actual {actual}")
    assert ok, f'check failed: {name} (expected {expected}, actual {actual})'
def info(name, value):
    REPORT.append({'check': name, 'expected': None, 'actual': value, 'pass': True, 'status': 'info'}); print(f'INFO  {name}: {value}')
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
def sha_text(s): return hashlib.sha256(s.encode()).hexdigest()
def state_sha(m):
    h = hashlib.sha256()
    for k, v in m.state_dict().items(): h.update(k.encode()); h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def rel(p): return Path(p).relative_to(ROOT).as_posix()
print(f'all inputs found; TestLog has {len(_log0)} entr(ies); device', DEVICE, GPU or '')

## 2. Fixed code (hashed)

All code that turns crops into numbers is kept in `CODE` and run from there, so its sha256 can be recorded on val, checked again in the preflight and written into the TestLog entry. `model_class`, `MEAN`/`STD` and `gpu_eval` are copied verbatim from `train_cls_2026-10-25.ipynb`; `evaluation` is new in P04 and is used unchanged for val and test.

In [ ]:
CODE = {}
CODE['model_class'] = r'''
class Pretrained(nn.Module):
    def __init__(self, n_classes, weights, mode, dropout):
        super().__init__()
        net = torchvision.models.resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(net.fc.in_features, n_classes))
        self.mode = mode
        if mode == 'frozen':
            for p in self.backbone.parameters(): p.requires_grad = False
    def train(self, on=True):
        super().train(on)
        if self.mode == 'frozen': self.backbone.eval()       # BatchNorm statistics stay fixed
        return self
    def forward(self, x): return self.head(self.backbone(x))
'''
CODE['preprocessing'] = r'''
class ac:   # stands in for aerial_crops.py (train_cls_2026-10-25.ipynb); the next line is copied from it
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, used for scratch and pretrained alike
MEAN = torch.tensor(ac.MEAN).view(1, 3, 1, 1); STD = torch.tensor(ac.STD).view(1, 3, 1, 1)

def gpu_eval(x, out):
    x = x.permute(0, 3, 1, 2).float() / 255
    if x.shape[-1] != out: x = F.interpolate(x, size=(out, out), mode='bilinear', antialias=True, align_corners=False)
    return (x - MEAN.to(x.device)) / STD.to(x.device)
'''

In [ ]:
CODE['evaluation'] = r'''
K = len(CLASSES); LABELS = list(range(K)); CID = {c: i for i, c in enumerate(CLASSES)}

def size_px(df):                       # sqrt(w * h) in original-frame pixels, from the CropIndex box
    return np.sqrt(df.w.to_numpy(float) * df.h.to_numpy(float))

def size_bin(s):
    return np.where(s < SIZE_EDGES[0], 'small', np.where(s < SIZE_EDGES[1], 'medium', 'large'))

@torch.no_grad()
def logits_fp32(X):                    # raw logits, fp32, no AMP, no logit bias; model in eval mode
    assert not model.training
    out = np.empty((len(X), K), np.float32)
    for i in range(0, len(X), BATCH):
        out[i:i + BATCH] = model(gpu_eval(torch.from_numpy(np.ascontiguousarray(X[i:i + BATCH])).to(DEVICE), IMG)).float().cpu().numpy()
    return out

def macro_f1_all(Y, pred):             # as train_cls metrics(): all 8 labels
    return float(f1_score(Y, pred, labels=LABELS, average='macro', zero_division=0))

def shown(value, n):                   # what the report shows for a per-class number
    return 'n=0' if n == 0 else ('n<30' if n < MIN_N else f'{value:.4f}')

def tables(split, labels, Y, pred, bins, subsets):
    # tidy rows: one per (dataset, size_bin, metric, class); value is empty where display is n=0 / n<30
    rows = []
    for ds, dmask in subsets.items():
        for b in BINS + ['all']:
            m = dmask if b == 'all' else dmask & (bins == b)
            y, p = Y[m], pred[m]; n = np.bincount(y, minlength=K)
            base = {'split': split, 'labels': labels, 'dataset': ds, 'size_bin': b}
            present = [c for c in LABELS if n[c] > 0]
            mf = float(f1_score(y, p, labels=present, average='macro', zero_division=0)) if present else np.nan
            small = [CLASSES[c] for c in present if n[c] < MIN_N]
            rows.append({**base, 'metric': 'macro_f1', 'class': '', 'support': int(m.sum()), 'value': mf,
                         'display': 'n=0' if not present else f'{mf:.4f}',
                         'note': f'over the {len(present)} classes present' + (f'; n<30: {", ".join(small)}' if small else '')})
            rec = recall_score(y, p, labels=LABELS, average=None, zero_division=0) if len(y) else np.zeros(K)
            f1 = f1_score(y, p, labels=LABELS, average=None, zero_division=0) if len(y) else np.zeros(K)
            for c in LABELS:
                rows.append({**base, 'metric': 'support', 'class': CLASSES[c], 'support': int(n[c]), 'value': int(n[c]),
                             'display': str(int(n[c])), 'note': ''})
                for metric, v in (('recall', rec[c]), ('f1', f1[c])):
                    ok = n[c] >= MIN_N
                    rows.append({**base, 'metric': metric, 'class': CLASSES[c], 'support': int(n[c]),
                                 'value': float(v) if ok else np.nan, 'display': shown(v, n[c]), 'note': ''})
            for src, dst in FOCUS:
                c = CID[src]; k = int(((y == c) & (p == CID[dst])).sum())
                v = k / n[c] if n[c] else np.nan
                rows.append({**base, 'metric': f'{src}->{dst}', 'class': src, 'support': int(n[c]),
                             'value': float(v) if n[c] >= MIN_N else np.nan, 'display': shown(v, n[c]),
                             'note': f'{k} of {int(n[c])} {src} crops predicted {dst}'})
    return pd.DataFrame(rows)

def compact(T, metric):                # print helper: display strings, rows = dataset / class, columns = size bins
    t = T[T.metric == metric]
    idx = ['labels', 'dataset'] + ([] if metric == 'macro_f1' else ['class'])
    return t.pivot_table(index=idx, columns='size_bin', values='display', aggfunc='first', sort=False)[BINS + ['all']]
'''
EVAL_FUNCS = ['size_px', 'size_bin', 'logits_fp32', 'macro_f1_all', 'shown', 'tables', 'compact']
for part in ('model_class', 'preprocessing', 'evaluation'): exec(CODE[part])
FROZEN = {n: globals()[n] for n in EVAL_FUNCS + ['gpu_eval', 'Pretrained']}
CODE_SHA = sha_text(json.dumps(CODE, sort_keys=True))
print('code sha256:', CODE_SHA)

## 3. Checkpoint (weights_only=True) and model

In [ ]:
CKPT_SHA = sha256(P['ckpt'])
ck = torch.load(P['ckpt'], map_location='cpu', weights_only=True)
check('checkpoint has model, classes, img_size', [], sorted({'model', 'classes', 'img_size'} - set(ck)))
check('checkpoint classes == CLASSES', CLASSES, list(ck['classes']))
check('checkpoint img_size', EXPECT['img_size'], int(ck['img_size']))
IMG = int(ck['img_size'])
for k, v in (('seed', 43), ('lr', 0.002)):
    if k in ck: check(f'checkpoint {k}', v, ck[k])
    else: print(f'checkpoint stores no {k!r}')
if 'crops_used' in ck:                 # the crop files this checkpoint was trained / evaluated with
    for k, f in (('val', IN['val_npy']), ('test', IN_TEST['test_npy']), ('index', IN['crop_index'])):
        if k in ck['crops_used']: check(f'checkpoint crops_used[{k!r}] is the file used here', Path(f).name, ck['crops_used'][k])
else: print("checkpoint stores no 'crops_used'")
print('logit_bias stored:', 'logit_bias' in ck, '-> NOT applied (raw logits only)')
model = Pretrained(8, weights=None, mode='full', dropout=0.2)
model.load_state_dict(ck['model'], strict=True)
model = model.eval().to(DEVICE)
STATE_SHA = state_sha(model); del ck
print(f'loaded (strict), eval mode; weights sha256 {STATE_SHA[:16]}...')

## Part 1 · Validation (test is not opened)

### 4. Val reproduction (fp32)
Validation rows of CropIndex, read in chunks so that no test row is kept; ordered by `row` as in `train_cls`.

In [ ]:
VAL = pd.concat([ch[ch.split == 'val'] for ch in
                 pd.read_csv(P['crop_index'], keep_default_na=False, dtype={'ann_id': str}, chunksize=20_000)])
VAL = VAL.sort_values('row').reset_index(drop=True)
check('val rows in CropIndex', EXPECT['val_crops'], len(VAL))
check('val row == 0..n-1', True, bool((VAL.row.to_numpy() == np.arange(len(VAL))).all()))
XV = np.load(P['val_npy'])
check('CropsVal shape', (EXPECT['val_crops'], EXPECT['crop_side'], EXPECT['crop_side'], 3), tuple(XV.shape))
check('CropsVal dtype', 'uint8', str(XV.dtype))
YV = VAL.category.map(CID)
check('every val category is a class', 0, int(YV.isna().sum())); YV = YV.to_numpy().astype(int)
LV = logits_fp32(XV); PV = LV.argmax(1); del XV
F1_V = macro_f1_all(YV, PV)
check(f'val macro-F1 = {EXPECT["val_macro_f1"]} +- {EXPECT["val_macro_f1_tol"]} (fp32; else checkpoint or preprocessing differ)',
      f'|diff| <= {EXPECT["val_macro_f1_tol"]}', f'{F1_V:.6f} (diff {F1_V - EXPECT["val_macro_f1"]:+.6f})',
      ok=abs(F1_V - EXPECT['val_macro_f1']) <= EXPECT['val_macro_f1_tol'])
info('val accuracy (fp32)', round(float((PV == YV).mean()), 6))

### 5. Size bins on val: counts

In [ ]:
def add_size(df, name):
    w, h = df.w.to_numpy(float), df.h.to_numpy(float)
    check(f'{name}: box w and h finite and > 0', 0, int((~np.isfinite(w) | ~np.isfinite(h) | (w <= 0) | (h <= 0)).sum()))
    return df.assign(size_px=size_px(df), size_bin=size_bin(size_px(df)))   # replaces Step 4's size_bin column (box_side bins)
VAL = add_size(VAL, 'val'); BV = VAL.size_bin.to_numpy()
check('val datasets', ['SeaDronesSee', 'UAVDT'], sorted(VAL.dataset.unique()))
CNT_V = pd.crosstab([VAL.dataset, VAL.category], VAL.size_bin).reindex(columns=BINS, fill_value=0)
CNT_V['all'] = CNT_V.sum(1)
print(f'val crops per size bin (sqrt(w*h): small < {SIZE_EDGES[0]} <= medium < {SIZE_EDGES[1]} <= large):')
print(pd.crosstab(VAL.dataset, VAL.size_bin, margins=True, margins_name='all').reindex(columns=BINS + ['all']).to_string())
print('\nper dataset and class:'); print(CNT_V.to_string())

### 6. Metrics by size on val

In [ ]:
SUB_V = {'SeaDronesSee': (VAL.dataset == 'SeaDronesSee').to_numpy(), 'UAVDT': (VAL.dataset == 'UAVDT').to_numpy(),
         'all': np.ones(len(VAL), bool)}
TV = tables('val', 'original', YV, PV, BV, SUB_V)
_mf = TV[(TV.metric == 'macro_f1') & (TV.dataset == 'all') & (TV.size_bin == 'all')].value.iloc[0]
check('val table: all / all macro-F1 equals the reproduction', round(F1_V, 10), round(float(_mf), 10))
_s = TV[TV.metric == 'support'].pivot_table(index=['dataset', 'class'], columns='size_bin', values='support', aggfunc='sum')
check('val table: small + medium + large == all, every dataset and class', True, bool((_s[BINS].sum(1) == _s['all']).all()))
print('macro-F1:'); print(compact(TV, 'macro_f1').to_string())
print('\nrecall:'); print(compact(TV, 'recall').to_string())
print('\nF1:'); print(compact(TV, 'f1').to_string())
for src, dst in FOCUS: print(f'\n{src}->{dst} (share of {src} crops):'); print(compact(TV, f'{src}->{dst}').to_string())

### 7. Output folder (never overwritten); `size_bins_val.csv` is written before test is opened

In [ ]:
TODAY = time.strftime('%Y-%m-%d')
P04_DIR = ROOT / '02_Classification/P04'; P04_DIR.mkdir(parents=True, exist_ok=True)
_done = [int(m.group(1)) for f in os.listdir(P04_DIR) if (m := re.fullmatch(rf'{TODAY}_SizeBins_v(\d+)', f))]   # _INCOMPLETE does not count
VERSION = max(_done, default=0) + 1
FINAL = P04_DIR / f'{TODAY}_SizeBins_v{VERSION}'
OUT = P04_DIR / (FINAL.name + '_INCOMPLETE')
if OUT.exists():                                   # never deleted automatically
    print(f'a failed run left {OUT.name}: inspect and delete it manually, then re-run')
    raise FileExistsError(f'{rel(OUT)} exists')
assert not FINAL.exists()
OUT.mkdir(); OUT_REL = rel(FINAL)
TV.to_csv(OUT / 'size_bins_val.csv', index=False)
VAL_CSV_SHA = sha256(OUT / 'size_bins_val.csv')
print('output folder:', OUT_REL, '(written as', OUT.name, 'until all checks pass); size_bins_val.csv written')

## Part 2 · Test, one opening (report only)

### 8. Preflight: nothing from test crops or predictions is read here
Reads the test rows of CropIndex (labels and boxes only) and the label-fix list, checks that code and weights are unchanged, and prints the plan. The TestLog entry is written in section 9, right before the first forward pass.

In [ ]:
# --- label-fix list: Step 4 audit (blind, before any prediction) ---
_aud = sorted((f for f in os.listdir(P['audit_dir']) if re.fullmatch(r'\d{4}-\d{2}-\d{2}_AuditFilled_v\d+\.csv', f)),
              key=lambda f: (f[:10], int(re.search(r'_v(\d+)\.csv$', f)[1])))
if not _aud:
    raise FileNotFoundError(f'no *_AuditFilled_vN.csv in {IN["audit_dir"]}: no label-fix list, stop and tell Abdulmohsen')
AUDIT_FILE = f'{IN["audit_dir"]}/{_aud[-1]}'
AF = pd.read_csv(ROOT / AUDIT_FILE, keep_default_na=False, dtype=str)
AF['verdict'] = AF.verdict.str.strip().str.lower(); AF['true_label'] = AF.true_label.str.strip()
check('audit sheet has the Step 4 columns', [], sorted({'audit_id', 'crop_id', 'split', 'category', 'verdict', 'true_label', 'note'} - set(AF.columns)))
check('audit verdicts are known', [], sorted(set(AF.verdict) - {'ok', 'wrong_class', 'not_object', 'bad_box', 'unsure'}))
AFT = AF[AF.split == 'test']
info('audit file', AUDIT_FILE); info('audited crops in test (all verdicts)', int(len(AFT)))
info('test verdicts', AFT.verdict.value_counts().to_dict())
if not len(AFT):
    raise RuntimeError('the audit sheet has no test rows: no label-fix list, stop and tell Abdulmohsen (nothing is guessed)')

TEST = pd.concat([ch[ch.split == 'test'] for ch in
                  pd.read_csv(P['crop_index'], keep_default_na=False, dtype={'ann_id': str}, chunksize=20_000)])
TEST = TEST.sort_values('row').reset_index(drop=True)
check('test row == 0..n-1', True, bool((TEST.row.to_numpy() == np.arange(len(TEST))).all()))
TEST['crop_id'] = TEST.crop_id.astype(str)
YT = TEST.category.map(CID)
check('every test category is a class', 0, int(YT.isna().sum())); YT = YT.to_numpy().astype(int)
TEST = add_size(TEST, 'test'); BT = TEST.size_bin.to_numpy()

LF = AFT[AFT.verdict.isin(['wrong_class', 'not_object'])].copy()
_ix = TEST.set_index('crop_id')
check('every audited test crop_id is a test row of CropIndex', [], sorted(set(AFT.crop_id) - set(_ix.index)))
check('audit category == CropIndex category (audited test rows)', 0, int((AFT.category.to_numpy() != _ix.loc[AFT.crop_id, 'category'].to_numpy()).sum()))
check('wrong_class rows: true_label is a class and differs from the label', 0,
      int((LF.verdict.eq('wrong_class') & (~LF.true_label.isin(CLASSES) | LF.true_label.eq(LF.category))).sum()))
LF = pd.DataFrame({'crop_id': LF.crop_id, 'dataset': _ix.loc[LF.crop_id, 'dataset'].to_numpy(),
                   'source': _ix.loc[LF.crop_id, 'source'].to_numpy(), 'image': _ix.loc[LF.crop_id, 'image'].to_numpy(),
                   'original_label': LF.category,
                   'corrected_label': np.where(LF.verdict == 'wrong_class', LF.true_label, 'EXCLUDED (not_object)'),
                   'verdict': LF.verdict, 'method': 'Step 4 blind label audit (stratified sample of 300, all splits), before any model prediction',
                   'discovered_after_predictions': False, 'source_file': AUDIT_FILE, 'audit_id': LF.audit_id, 'note': LF.note})
if GALLERY_FIX is not None:            # only by decision: found in the Step 8 error gallery, i.e. AFTER seeing predictions
    G = pd.read_csv(ROOT / GALLERY_FIX, keep_default_na=False, dtype=str)
    check('GALLERY_FIX columns', [], sorted({'crop_id', 'corrected_label', 'how'} - set(G.columns)))
    check('GALLERY_FIX crop_ids are test rows and not in the audit list', [], sorted((set(G.crop_id) - set(_ix.index)) | (set(G.crop_id) & set(LF.crop_id))))
    check('GALLERY_FIX corrected_label is a class or not_object', [], sorted(set(G.corrected_label) - set(CLASSES) - {'not_object'}))
    LF = pd.concat([LF, pd.DataFrame({'crop_id': G.crop_id, 'dataset': _ix.loc[G.crop_id, 'dataset'].to_numpy(),
        'source': _ix.loc[G.crop_id, 'source'].to_numpy(), 'image': _ix.loc[G.crop_id, 'image'].to_numpy(),
        'original_label': _ix.loc[G.crop_id, 'category'].to_numpy(),
        'corrected_label': np.where(G.corrected_label == 'not_object', 'EXCLUDED (not_object)', G.corrected_label),
        'verdict': np.where(G.corrected_label == 'not_object', 'not_object', 'wrong_class'),
        'method': 'Step 8 error gallery: ' + G.how, 'discovered_after_predictions': True, 'source_file': GALLERY_FIX,
        'audit_id': '', 'note': ''})], ignore_index=True)
LF = LF.reset_index(drop=True)
check('label-fix crop_ids unique', True, bool(LF.crop_id.is_unique))
BIASED = bool(LF.discovered_after_predictions.any())
print(f'\nLABEL-FIX LIST ({len(LF)} test crops; {int((LF.verdict == "not_object").sum())} excluded as not_object):')
print(LF.drop(columns=['source_file', 'image']).to_string(index=False) if len(LF) else '  (none: the audited test crops were all ok / bad_box / unsure)')

# corrected labels: wrong_class -> true label; not_object -> removed. Every other crop keeps its original label.
YT_C = YT.copy(); KEEP_C = np.ones(len(TEST), bool)
_pos = pd.Series(np.arange(len(TEST)), index=TEST.crop_id)
for r in LF.itertuples():
    if r.verdict == 'not_object': KEEP_C[_pos[r.crop_id]] = False
    else: YT_C[_pos[r.crop_id]] = CID[r.corrected_label]

# --- everything else that can fail without looking at test ---
_n = os.path.getsize(P['test_npy']) - len(TEST) * EXPECT['crop_side'] ** 2 * 3   # .npy header only; the file is not opened
check('CropsTest file size fits len(test rows) x 96x96x3 uint8 (header only left)', True, 0 < _n <= 4096)
check('code unchanged since val (sha256)', CODE_SHA, sha_text(json.dumps(CODE, sort_keys=True)))
check('evaluation functions are the ones run on val', [], [n for n, f in FROZEN.items() if globals()[n] is not f])
check('model weights unchanged since load (sha256)', STATE_SHA, state_sha(model))
check('model in eval mode', False, model.training)
check('size_bins_val.csv unchanged', VAL_CSV_SHA, sha256(OUT / 'size_bins_val.csv'))
check('every check so far passed', True, all(r['pass'] for r in REPORT))
LOG_BYTES = P['test_log'].read_bytes()                               # the file as it is before the entry
LOG = pd.read_csv(io.BytesIO(LOG_BYTES), dtype=str, keep_default_na=False)
print(f'\nTestLog: {len(LOG)} entr(ies), {len(LOG_BYTES)} bytes'); print(LOG.to_string(index=False))
check('TestLog header has the columns this notebook fills (none is added)', [], [c for c in LOG_COLS if c not in LOG.columns])
check('this opening is not logged yet', False, REASON in set(LOG['reason']))
check('earlier openings of test in TestLog', EXPECT['test_log_entries_before'], len(LOG))
IS_RGB = ((TEST.dataset == EXCLUDE_RGB[0]) & (TEST.source == EXCLUDE_RGB[1])).to_numpy()
SUB_T = {'SeaDronesSee': (TEST.dataset == 'SeaDronesSee').to_numpy(), 'UAVDT': (TEST.dataset == 'UAVDT').to_numpy(),
         'all': np.ones(len(TEST), bool), 'all without SDS rgb': ~IS_RGB}
print(f"""
PLAN (fixed now; nothing below depends on a test result):
  model   {IN['ckpt']} (sha256 {CKPT_SHA[:16]}...), raw logits, fp32, no logit bias, no threshold
  data    {len(TEST):,} test crops ({IN_TEST['test_npy']}), {int(IS_RGB.sum()):,} of them SeaDronesSee 'rgb'
  check   argmax == TestPredictions_v1 column fullFT_s43_raw, except near-ties (top1 - top2 < {TIE_GAP}); otherwise stop
  tables  tables() as on val, size bins small/medium/large/all x datasets {list(SUB_T)}:
          macro-F1, per-class recall / F1 / support, truck->car and bus->car; n<{MIN_N} shown as 'n<30'
  labels  original (headline) and corrected ({int((LF.verdict == 'wrong_class').sum())} relabelled, {int((LF.verdict == 'not_object').sum())} removed);
          found after predictions: {int(LF.discovered_after_predictions.sum())} -> corrected numbers {'BIASED UPWARD' if BIASED else 'not biased by predictions'}
  outputs size_bins_test.csv, label_fix.csv, recall_by_size.png, p04_report.json in {OUT_REL}
  log     TestLog entry: reason '{REASON}', code sha256 {CODE_SHA[:16]}...""")

### 9. TestLog entry, then the one opening of test
This cell writes the TestLog entry first; only then are the test crops and `TestPredictions_v1` allowed through the guard.

In [ ]:
# one row appended, never rewritten; columns in the header's order; other columns empty
check('TestLog unchanged since the preflight (bytes)', True, P['test_log'].read_bytes() == LOG_BYTES)
entry = {c: '' for c in LOG.columns}
entry.update({'when': time.strftime('%Y-%m-%d %H:%M'), 'checkpoints': Path(IN['ckpt']).name, 'reason': REASON,
              'notebook_version': f'{NOTEBOOK} code_sha256={CODE_SHA}'})
EOL = '\r\n' if b'\r\n' in LOG_BYTES else '\n'                       # same line ending as the existing file
NEWLINE_ADDED = not LOG_BYTES.endswith(b'\n')                         # the last line had no newline: add it before the entry
_row = io.StringIO(); csv.writer(_row, lineterminator=EOL).writerow([entry[c] for c in LOG.columns])
with open(P['test_log'], 'a', newline='') as f:
    if NEWLINE_ADDED: f.write(EOL)
    f.write(_row.getvalue())
_new = P['test_log'].read_bytes()
_back = pd.read_csv(io.BytesIO(_new), dtype=str, keep_default_na=False)
check('TestLog: the old file (bytes) is exactly the start of the new file', True, _new.startswith(LOG_BYTES))
check('TestLog: only the entry was added (bytes)', ((EOL if NEWLINE_ADDED else '') + _row.getvalue()).encode(), _new[len(LOG_BYTES):])
check('TestLog: exactly one row more, old rows unchanged', [len(LOG) + 1, True],
      [len(_back), list(_back.columns) == list(LOG.columns) and _back.iloc[:len(LOG)].equals(LOG)])
check('TestLog: the new row is the entry', [entry[c] for c in LOG.columns], _back.iloc[-1].tolist())
LOG_ENTRY = dict(entry)
LOG_WRITE = {'mode': 'append, one row', 'line_ending': repr(EOL), 'newline_added_before_entry': NEWLINE_ADDED,
             'bytes_before': len(LOG_BYTES), 'bytes_after': len(_new),
             'sha256_before': hashlib.sha256(LOG_BYTES).hexdigest(), 'sha256_after': hashlib.sha256(_new).hexdigest()}
if NEWLINE_ADDED: info('TestLog had no newline at the end: one was added before the entry', True)
print(f'TestLog now has {len(_back)} entries:'); print(_back.to_string(index=False))

ALLOWED |= {os.path.abspath(P['test_npy']), os.path.abspath(P['test_pred'])}   # opened from here on
XT = np.load(P['test_npy'])
check('CropsTest shape', (len(TEST), EXPECT['crop_side'], EXPECT['crop_side'], 3), tuple(XT.shape))
check('CropsTest dtype', 'uint8', str(XT.dtype))
LT = logits_fp32(XT); PT = LT.argmax(1); del XT

PRED = pd.read_csv(P['test_pred'], keep_default_na=False, dtype={'crop_id': str})
check('test crop order == TestPredictions_v1', True, TEST.crop_id.tolist() == PRED.crop_id.tolist())
check('TestPredictions_v1 labels == CropIndex labels', True, bool((PRED.category.to_numpy() == TEST.category.to_numpy()).all()))
_want = PRED['fullFT_s43_raw'].map(CID)
check('fullFT_s43_raw values are classes', 0, int(_want.isna().sum())); _want = _want.to_numpy().astype(int)
_top2 = np.sort(LT, 1)[:, -2:]; _gap = _top2[:, 1] - _top2[:, 0]
_diff = PT != _want; _tie = _diff & (_gap < TIE_GAP)
info(f'argmax differs from Step 8 (fp16) only at near-ties (gap < {TIE_GAP}): count', int(_tie.sum()))
check('argmax == TestPredictions_v1 fullFT_s43_raw except near-ties', 0, int((_diff & ~_tie).sum()))
F1_T_STEP8 = macro_f1_all(YT, _want); F1_T = macro_f1_all(YT, PT)
info('test macro-F1, all 8 labels: this run fp32 / Step 8 file', [round(F1_T, 6), round(F1_T_STEP8, 6)])

### 10. Test tables: original labels (headline) and corrected labels

In [ ]:
TT_O = tables('test', 'original', YT, PT, BT, SUB_T)
TT_C = tables('test', 'corrected', YT_C[KEEP_C], PT[KEEP_C], BT[KEEP_C], {k: v[KEEP_C] for k, v in SUB_T.items()})
TT = pd.concat([TT_O, TT_C], ignore_index=True)
_mf = TT_O[(TT_O.metric == 'macro_f1') & (TT_O.dataset == 'all') & (TT_O.size_bin == 'all')].value.iloc[0]
check('test table: all / all macro-F1 (original) equals the 8-label value', round(F1_T, 10), round(float(_mf), 10)) \
    if (np.bincount(YT, minlength=K) > 0).all() else info('test: a class is absent, macro over present classes', float(_mf))
for name, T in (('original', TT_O), ('corrected', TT_C)):
    _s = T[T.metric == 'support'].pivot_table(index=['dataset', 'class'], columns='size_bin', values='support', aggfunc='sum')
    check(f'test table ({name}): small + medium + large == all', True, bool((_s[BINS].sum(1) == _s['all']).all()))
check('corrected table: test crops kept', int(len(TEST) - (LF.verdict == 'not_object').sum()),
      int(TT_C[(TT_C.metric == 'macro_f1') & (TT_C.dataset == 'all') & (TT_C.size_bin == 'all')].support.iloc[0]))
CNT_T = pd.crosstab([TEST.dataset, TEST.category], TEST.size_bin).reindex(columns=BINS, fill_value=0); CNT_T['all'] = CNT_T.sum(1)
print('test crops per dataset, class and size bin:'); print(CNT_T.to_string())
print('\nmacro-F1:'); print(compact(TT, 'macro_f1').to_string())
print('\nrecall:'); print(compact(TT, 'recall').to_string())
print('\nF1:'); print(compact(TT, 'f1').to_string())
for src, dst in FOCUS: print(f'\n{src}->{dst} (share of {src} crops):'); print(compact(TT, f'{src}->{dst}').to_string())
TT.to_csv(OUT / 'size_bins_test.csv', index=False)
LF.to_csv(OUT / 'label_fix.csv', index=False)

### 11. Figure: per-class recall by size, val and test (original labels) side by side
A cell with fewer than 30 crops has no bar and is marked `n<30` (or `n=0`).

In [ ]:
SEQ = {'small': '#86b6ef', 'medium': '#2a78d6', 'large': '#104281'}      # one hue, light -> dark = small -> large
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), sharey=True)
for ax, (T, title) in zip(axes, ((TV, f'val ({len(VAL):,} crops)'), (TT_O, f'test, original labels ({len(TEST):,} crops)'))):
    t = T[(T.metric == 'recall') & (T.dataset == 'all')].set_index(['class', 'size_bin'])
    xs = np.arange(K); wd = 0.26
    for j, b in enumerate(BINS):
        for i, c in enumerate(CLASSES):
            r = t.loc[(c, b)]; x = xs[i] + (j - 1) * (wd + 0.02)
            if np.isfinite(r.value): ax.bar(x, r.value, wd, color=SEQ[b], label=None)
            else: ax.text(x, 0.02, r.display, rotation=90, ha='center', va='bottom', fontsize=7, color='#555555')
    ax.set_xticks(xs); ax.set_xticklabels([c.replace('life_saving_appliances', 'LSA') for c in CLASSES], fontsize=9)
    ax.set_title(title, fontsize=10, loc='left'); ax.set_ylim(0, 1.05); ax.set_xlim(-0.6, K - 0.4)
    ax.grid(axis='y', color='#e6e6e6', lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
axes[0].set_ylabel('recall')
from matplotlib.patches import Patch
fig.legend([Patch(color=SEQ[b]) for b in BINS],
           [f'small (< {SIZE_EDGES[0]} px)', f'medium ({SIZE_EDGES[0]} to < {SIZE_EDGES[1]} px)', f'large (>= {SIZE_EDGES[1]} px)'],
           loc='upper center', ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f'Per-class recall by object size, sqrt(w*h) in frame px; cls-full-64-s43, raw logits; no bar = fewer than {MIN_N} crops',
             fontsize=10, y=0.94)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig(OUT / 'recall_by_size.png', dpi=150); plt.show()

## 12. Report, hygiene, rename

In [ ]:
def mf_table(T):
    t = T[T.metric == 'macro_f1']
    return {f'{l} | {d}': {b: (None if not np.isfinite(v) else round(float(v), 4)) for b, v in zip(g.size_bin, g.value)}
            for (l, d), g in t.groupby(['labels', 'dataset'], sort=False)}
LABEL_FIX_INFO = {
    'source_files': sorted(set(LF.source_file)) or [AUDIT_FILE], 'audited_test_crops': int(len(AFT)),
    'relabelled': int((LF.verdict == 'wrong_class').sum()), 'excluded_not_object': int((LF.verdict == 'not_object').sum()),
    'found_after_predictions': int(LF.discovered_after_predictions.sum()), 'corrected_numbers_biased_upward': BIASED,
    'note': ('Some corrections were found in the Step 8 error gallery, after seeing model predictions: the corrected numbers are '
             'biased upward; the original-label numbers are the headline.' if BIASED else
             'All corrections come from the blind Step 4 audit (before any prediction). The audit is a sample, so only audited test '
             'crops are corrected; all other test crops keep their original labels. The original-label numbers are the headline.')}
REP = {'created': time.strftime('%Y-%m-%d %H:%M'), 'output': OUT_REL, 'notebook': NOTEBOOK,
       'model': {'checkpoint': IN['ckpt'], 'sha256': CKPT_SHA, 'weights_sha256': STATE_SHA, 'onnx_delivered_as': '2026-10-31_ClsResNet18_s43_v1',
                 'raw_logits_no_logit_bias': True, 'precision': 'fp32, TF32 off'},
       'size_definition': {'size': 'sqrt(w * h), px in the original frame (CropIndex w, h)', 'edges': list(SIZE_EDGES), 'bins': BINS,
                           'min_n_per_class_cell': MIN_N},
       'code_sha256': CODE_SHA, 'val': {'n_crops': len(VAL), 'macro_f1_fp32': F1_V, 'expected': EXPECT['val_macro_f1'],
                                        'macro_f1_by_size': mf_table(TV)},
       'test': {'n_crops': len(TEST), 'n_sds_rgb': int(IS_RGB.sum()), 'macro_f1_fp32_all_labels': F1_T,
                'macro_f1_step8_file_s43_raw': F1_T_STEP8, 'near_tie_flips_vs_step8': int(_tie.sum()),
                'macro_f1_by_size': mf_table(TT), 'headline': 'original labels; primary dataset row: all without SDS rgb (25 Oct decision)'},
       'test_log_entry': LOG_ENTRY, 'test_log_write': LOG_WRITE, 'label_fix': LABEL_FIX_INFO,
       'rules': 'test opened once for this report; no threshold, model choice or other decision is based on test',
       'versions': VERSIONS, 'inputs': {**IN, **IN_TEST, 'GALLERY_FIX': GALLERY_FIX, 'audit_file': AUDIT_FILE}}
BAD = ['/content/', '/Users/', 'MyDrive']
def report_json():
    return json.dumps({**REP, 'checks': REPORT, 'all_pass': all(r['pass'] for r in REPORT)}, indent=1, default=str)
for f in ('size_bins_val.csv', 'size_bins_test.csv', 'label_fix.csv'):
    check(f'no absolute paths in {f}', [], [b for b in BAD if b in (OUT / f).read_text()])
check('no absolute paths in p04_report.json', [], [b for b in BAD if b in report_json()])   # scanned before writing
RJ = report_json(); assert not any(b in RJ for b in BAD)
(OUT / 'p04_report.json').write_text(RJ)
check('output files', ['label_fix.csv', 'p04_report.json', 'recall_by_size.png', 'size_bins_test.csv', 'size_bins_val.csv'],
      sorted(p.name for p in OUT.iterdir()))
assert all(r['pass'] for r in REPORT) and not FINAL.exists()
OUT.rename(FINAL)
print('renamed ->', OUT_REL)

## 13. Summary

In [ ]:
print(f'val macro-F1 (fp32): {F1_V:.4f} (expected {EXPECT["val_macro_f1"]}) | test macro-F1, original labels: {F1_T:.4f}')
print(f"label fix: {LABEL_FIX_INFO['relabelled']} relabelled, {LABEL_FIX_INFO['excluded_not_object']} excluded "
      f"({LABEL_FIX_INFO['audited_test_crops']} audited test crops); biased upward: {BIASED}")
S = pd.concat([TV, TT]); S = S[S.metric == 'macro_f1']
print('\nmacro-F1 by size bin:')
print(S.pivot_table(index=['split', 'labels', 'dataset'], columns='size_bin', values='display', aggfunc='first', sort=False)[BINS + ['all']].to_string())
print(); print(pd.DataFrame([{'check': r['check'], 'result': 'INFO' if r.get('status') == 'info' else ('PASS' if r['pass'] else 'FAIL')}
                             for r in REPORT]).to_string(index=False))
n_info = sum(r.get('status') == 'info' for r in REPORT)
print(f"\nALL {len(REPORT) - n_info} CHECKS PASS ({n_info} info) -> {OUT_REL}")